# IMDb Plot → Genre Classifier (DistilBERT)

Fine-tunes `distilbert-base-uncased` on plot summaries from [`kolodkin/imdb-wikipedia-enriched`](https://huggingface.co/datasets/kolodkin/imdb-wikipedia-enriched) to predict genres as a multi-label classification problem.

- **Labels**: top 15 IMDb genres + `Other`
- **Split**: 80 / 10 / 10, multi-label stratified, deduped on plot hash, seeded
- **Eval**: per-genre precision / recall / F1 on **train** and **test**, plus bootstrap CI on macro-F1
- **Output**: pushed to [`kolodkin/imdb-genre-distilbert`](https://huggingface.co/kolodkin/imdb-genre-distilbert) as the final step
- **Runtime**: ~15 min on a Colab free T4


## Install

In [ ]:
# transformers is pinned to the v5 major: v5 renamed several TrainingArguments
# (e.g. group_by_length -> train_sampling_strategy), so an unpinned install can
# break this notebook whenever a new major ships. Quotes keep the shell from
# treating < / > as redirection.
!pip install -q "transformers>=5,<6" datasets huggingface_hub accelerate iterative-stratification scikit-learn tabulate


## Imports + config

In [ ]:
import hashlib
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import f1_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)


def get_secret(name, default=None):
    """Fetch a value from os.environ first, then Colab userdata; default otherwise.

    Colab gotcha: the first access to a secret in a session triggers a
    "Grant access?" popup. If you miss/dismiss it, userdata.get returns None
    silently — re-run this cell and click "Grant access" when prompted.
    """
    import os

    value = os.environ.get(name)
    if value is not None:
        return value
    try:
        from google.colab import userdata
    except ImportError:
        return default
    try:
        value = userdata.get(name)
    except Exception:
        # SecretNotFoundError, NotebookAccessError, user denied, etc.
        return default
    return value if value is not None else default


def require_secret(name):
    """Like get_secret(name) but raises with a helpful Colab message if unset."""
    value = get_secret(name)
    if value is None:
        raise RuntimeError(
            f"Secret '{name}' not found. Set it via env var, or in Colab: open the "
            f"🔑 Secrets panel, add '{name}', toggle Notebook access on, and click "
            f"'Grant access' in the popup that appears when you re-run this cell."
        )
    return value


SEED = 42
MODEL_NAME = "distilbert-base-uncased"
DATASET_NAME = "kolodkin/imdb-wikipedia-enriched"
HF_REPO = "kolodkin/imdb-genre-distilbert"
TOP_K_GENRES = 15
MAX_LENGTH = 256
NUM_EPOCHS = 3
LR = 5e-5

# PUSH_TO_HUB must be explicitly "0" or "1" via env var or Colab secret. Missing
# → RuntimeError with setup instructions; non-numeric → ValueError. No silent
# defaults — a typo or missing config can't push or skip on accident.
PUSH_TO_HUB = bool(int(require_secret("PUSH_TO_HUB")))

# Ceiling batch size. Trainer's auto_find_batch_size=True (set in cell 18)
# starts here and halves on CUDA OOM until it fits, so this is the upper
# bound — the actual batch size is logged when training starts.
BATCH_SIZE = 128

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Device: {torch.cuda.get_device_name(0)}  ({gb:.1f} GB)")
print(f"BATCH_SIZE ceiling = {BATCH_SIZE}")
print(f"PUSH_TO_HUB = {PUSH_TO_HUB}")


## Authenticate to Hugging Face (optional)

Runs only when `PUSH_TO_HUB = True`. Both `PUSH_TO_HUB` and `HF_TOKEN` are resolved by `get_secret(name)` which checks `os.environ` first, then Colab `userdata`. If `HF_TOKEN` is found the cell calls `huggingface_hub.login(token=...)` non-interactively; otherwise it falls back to `notebook_login()`. Create a write token at https://huggingface.co/settings/tokens.


In [ ]:
if not PUSH_TO_HUB:
    print("PUSH_TO_HUB is False — skipping HF login.")
else:
    from huggingface_hub import login, notebook_login

    hf_token = get_secret("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("Authenticated to Hugging Face via HF_TOKEN secret.")
    else:
        print("HF_TOKEN secret not set — falling back to interactive notebook_login().")
        notebook_login()


## Load dataset

In [ ]:
ds = load_dataset(DATASET_NAME, split="train")
print(f"Rows: {len(ds):,}")
print(f"Columns: {ds.column_names}")
ds[0]


## Clean + dedupe

In [ ]:
def normalize_plot(s):
    return " ".join((s or "").lower().split())

def plot_hash(s):
    return hashlib.md5(normalize_plot(s).encode()).hexdigest()

df = ds.to_pandas()
n_raw = len(df)

# 1. drop short / empty plots
df = df[df["plot"].fillna("").str.len() >= 50]
n_after_plot = len(df)

# 2. drop rows with no genres
df = df[df["genres"].apply(lambda g: isinstance(g, (list, np.ndarray)) and len(g) > 0)]
n_after_genres = len(df)

# 3. drop duplicate plots (normalized + hashed)
df["_hash"] = df["plot"].map(plot_hash)
df = df.drop_duplicates(subset="_hash").drop(columns="_hash").reset_index(drop=True)
n_after_dedupe = len(df)

print(f"raw rows:             {n_raw:>7,}")
print(f"  - short/empty plot: {n_raw - n_after_plot:>7,}")
print(f"  - no genres:        {n_after_plot - n_after_genres:>7,}")
print(f"  - duplicate plots:  {n_after_genres - n_after_dedupe:>7,}")
print(f"kept:                 {n_after_dedupe:>7,}")


## Label space — top 15 genres + Other

In [ ]:
genre_counts = Counter(g for genres in df["genres"] for g in genres)
top = [g for g, _ in genre_counts.most_common(TOP_K_GENRES)]
print("Top genres:", top)

LABELS = top + ["Other"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
NUM_LABELS = len(LABELS)

def encode_labels(genres):
    vec = np.zeros(NUM_LABELS, dtype=np.float32)
    has_top = False
    for g in genres:
        if g in label2id:
            vec[label2id[g]] = 1.0
            has_top = True
    if not has_top:
        vec[label2id["Other"]] = 1.0
    return vec

df["labels"] = df["genres"].apply(encode_labels)

label_matrix = np.stack(df["labels"].values)
print("\nLabel frequencies:")
for label, count in zip(LABELS, label_matrix.sum(axis=0).astype(int)):
    print(f"  {label:<15} {count:>6,}")


## Stratified 80 / 10 / 10 split

In [ ]:
X = np.arange(len(df))
y = np.stack(df["labels"].values)

# 80 / 20 first
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, holdout_idx = next(msss.split(X, y))

# split 20 → 10 / 10
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
val_rel, test_rel = next(msss2.split(holdout_idx, y[holdout_idx]))
val_idx = holdout_idx[val_rel]
test_idx = holdout_idx[test_rel]

print(f"train: {len(train_idx):,}  |  val: {len(val_idx):,}  |  test: {len(test_idx):,}")
assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0


## Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf(idx):
    sub = df.iloc[idx][["plot", "labels"]].reset_index(drop=True)
    return Dataset.from_pandas(sub)

train_ds = to_hf(train_idx)
val_ds = to_hf(val_idx)
test_ds = to_hf(test_idx)

def tokenize(batch):
    return tokenizer(batch["plot"], truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["plot"])
val_ds = val_ds.map(tokenize, batched=True, remove_columns=["plot"])
test_ds = test_ds.map(tokenize, batched=True, remove_columns=["plot"])


## Model + Trainer

In [ ]:
import math

# LOAD REPORT is expected here: distilbert-base-uncased was pre-trained as a
# masked language model. AutoModelForSequenceClassification drops the MLM head
# (UNEXPECTED: vocab_transform / vocab_projector / vocab_layer_norm) and bolts
# on a fresh classification head (MISSING: pre_classifier / classifier), which
# trainer.train() in the next cell will tune. The model is unsafe for inference
# until that training step has run.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    return {
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_samples": f1_score(labels, preds, average="samples", zero_division=0),
    }

# warmup_ratio is deprecated for removal in transformers v5.2; compute the
# equivalent warmup_steps from train size, batch size, and epoch count so the
# 10% warmup tracks the actual schedule. Note: if auto_find_batch_size halves
# the effective batch on OOM, warmup_steps becomes proportionally short — the
# loss won't notice but the schedule is no longer exactly 10%.
total_train_steps = math.ceil(len(train_ds) / BATCH_SIZE) * NUM_EPOCHS
warmup_steps = int(0.1 * total_train_steps)
print(f"total train steps (at BATCH_SIZE ceiling): {total_train_steps:,}  |  warmup steps: {warmup_steps:,}")

args = TrainingArguments(
    output_dir="./out",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    auto_find_batch_size=True,  # halve per_device_train_batch_size on CUDA OOM until it fits
    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
    # group similar-length samples per batch → tighter dynamic padding, fewer wasted FLOPs on [PAD]
    train_sampling_strategy="group_by_length",
    dataloader_num_workers=2,   # overlap CPU tokenize/collate with GPU compute
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)


## Train

Per-epoch table columns:

- **Training Loss** — BCE loss averaged over recent training batches; should trend down.
- **Validation Loss** — BCE loss on the held-out val split; rising while training drops = overfitting.

The three F1 columns are all F1 scores (`2·P·R / (P+R)`) over the multi-label `N × 16` prediction grid — they differ only in **what they average over**:

| Metric | Aggregates over | Equal weight to | Answers |
|---|---|---|---|
| **F1 Micro** | every cell pooled together | each decision | "Across all genre-assignment decisions, how accurate are we?" — dominated by frequent genres (Drama, Comedy). |
| **F1 Macro** | per column (genre), then average the 16 | each genre | "How well do we do on the *average genre*, rare ones counting fully?" — **the model-selection metric** (`metric_for_best_model="f1_macro"`). |
| **F1 Samples** | per row (movie), then average over movies | each movie | "For the *average movie*, did we get its set of genres right?" |

A large gap between macro and micro/samples signals uneven per-genre performance (strong on common genres, weak on the long tail).

In [ ]:
trainer.train()


## Evaluation helper

Predicts on a tokenized split with the (now trained) best checkpoint and returns the raw label/prediction matrices plus a per-genre precision / recall / F1 table sorted by support. Used by the train and test evaluation cells below.

In [ ]:
def evaluate_split(ds_split, name):
    pred_out = trainer.predict(ds_split)
    probs = 1 / (1 + np.exp(-pred_out.predictions))
    preds = (probs >= 0.5).astype(int)
    labels_arr = pred_out.label_ids.astype(int)

    p, r, f1, support = precision_recall_fscore_support(
        labels_arr, preds, average=None, zero_division=0
    )
    per_genre = pd.DataFrame(
        {"genre": LABELS, "precision": p, "recall": r, "f1": f1, "support": support}
    ).sort_values("support", ascending=False)

    overall = {
        "f1_macro": f1_score(labels_arr, preds, average="macro", zero_division=0),
        "f1_micro": f1_score(labels_arr, preds, average="micro", zero_division=0),
        "f1_samples": f1_score(labels_arr, preds, average="samples", zero_division=0),
    }

    print(f"=== {name} ({len(labels_arr):,} rows) ===")
    for k, v in overall.items():
        print(f"  {k}: {v:.4f}")
    print()
    print(per_genre.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return overall, per_genre, labels_arr, preds


## Per-genre precision / recall on **train**

A useful sanity-check: if train P/R are near-perfect while test lags, we're overfitting. If train P/R are mediocre, the model is underfit and more epochs / capacity would help.

In [ ]:
train_overall, train_per_genre, _, _ = evaluate_split(train_ds, "TRAIN")


## Per-genre precision / recall on **test**

In [ ]:
test_overall, test_per_genre, test_labels, test_preds = evaluate_split(test_ds, "TEST")


## Train vs Test gap (per genre)

The `delta_f1` column is `train_f1 - test_f1`; large positive values indicate overfitting on that label.

In [ ]:
gap = (
    train_per_genre[["genre", "f1"]]
    .rename(columns={"f1": "train_f1"})
    .merge(
        test_per_genre[["genre", "f1", "support"]].rename(columns={"f1": "test_f1", "support": "test_support"}),
        on="genre",
    )
)
gap["delta_f1"] = gap["train_f1"] - gap["test_f1"]
gap = gap.sort_values("test_support", ascending=False)
print(gap.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


## Bootstrap 95% CI on test macro-F1

In [ ]:
rng = np.random.default_rng(SEED)
N = len(test_labels)
B = 1000
boots = np.empty(B)
for b in range(B):
    idx = rng.integers(0, N, size=N)
    boots[b] = f1_score(test_labels[idx], test_preds[idx], average="macro", zero_division=0)

lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"test macro-F1 = {test_overall['f1_macro']:.4f}  [95% CI {lo:.4f}, {hi:.4f}]")


## Improve #1 — per-class decision thresholds (validation-fit)

The model emits a probability per genre; the rest of the notebook turns those into yes/no labels with a blanket **0.5**. For imbalanced multi-label that's rarely optimal — rare genres often need a *lower* cutoff to be predicted at all. Here we **fit one threshold per genre on the validation split** (the value maximizing that genre's F1), then **apply the frozen thresholds to test**. Fitting on val and reporting on test keeps it honest — test is never used to choose cutoffs.

This is pure post-processing — **no retraining** — so it isolates how much headroom was just thresholding (and it's exactly the calibration the frozen-probe baseline got for free). The cell prints test F1 at the fixed 0.5 vs the tuned thresholds, plus the chosen threshold per genre.

In [ ]:
# Per-class threshold tuning. The blanket 0.5 cutoff is rarely optimal for
# imbalanced multi-label — rare genres often need a lower threshold to fire at
# all. We fit one threshold per genre on the VALIDATION split (the value that
# maximizes that genre's F1) and apply the frozen thresholds to TEST. Fitting on
# val and reporting on test keeps it honest — test is never used to pick cutoffs.
val_out = trainer.predict(val_ds)
val_probs = 1 / (1 + np.exp(-val_out.predictions))
val_labels = val_out.label_ids.astype(int)

test_out = trainer.predict(test_ds)
test_probs = 1 / (1 + np.exp(-test_out.predictions))
test_labels_thr = test_out.label_ids.astype(int)

# For each genre, sweep candidate thresholds and keep the best F1 on val. Fall
# back to 0.5 if no threshold beats zero F1 (e.g. a genre absent from val).
grid = np.linspace(0.05, 0.95, 19)
best_thresholds = np.full(NUM_LABELS, 0.5)
for j in range(NUM_LABELS):
    f1s = [f1_score(val_labels[:, j], (val_probs[:, j] >= t).astype(int), zero_division=0) for t in grid]
    best = int(np.argmax(f1s))
    if f1s[best] > 0:
        best_thresholds[j] = grid[best]

default_preds = (test_probs >= 0.5).astype(int)
tuned_preds = (test_probs >= best_thresholds).astype(int)  # broadcast (M,16) >= (16,)

def overall_f1(y_true, y_pred):
    return {
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_samples": f1_score(y_true, y_pred, average="samples", zero_division=0),
    }

default_overall = overall_f1(test_labels_thr, default_preds)
tuned_overall = overall_f1(test_labels_thr, tuned_preds)

thr_cmp = pd.DataFrame(
    {
        "metric": list(default_overall.keys()),
        "thresh_0.5": [default_overall[k] for k in default_overall],
        "tuned": [tuned_overall[k] for k in default_overall],
    }
)
thr_cmp["improvement"] = thr_cmp["tuned"] - thr_cmp["thresh_0.5"]

print("=== Fixed 0.5 vs per-class tuned thresholds (TEST) ===")
print(thr_cmp.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nTuned thresholds (fit on validation, sorted):")
thr_tbl = pd.DataFrame({"genre": LABELS, "threshold": best_thresholds}).sort_values("threshold")
print(thr_tbl.to_string(index=False, float_format=lambda x: f"{x:.2f}"))


## Try the model

Uses the in-memory best checkpoint (so this cell works before — or independently of — the final push).

In [ ]:
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    top_k=None,
    device=0 if torch.cuda.is_available() else -1,
)

samples = [
    "A young wizard discovers a magical school of witchcraft and faces a dark sorcerer.",
    "Two cops in 1970s Los Angeles investigate a string of brutal murders linked to organized crime.",
    "A robot from the future is sent back in time to protect a teenager from killer machines.",
    "A struggling stand-up comedian falls in love with a journalist while touring small clubs.",
]
for s in samples:
    out = clf(s)[0]
    top3 = sorted(out, key=lambda x: -x["score"])[:3]
    pretty = ", ".join(f"{x['label']} ({x['score']:.2f})" for x in top3)
    print(f"- {s}\n  → {pretty}\n")


## Did we actually improve? — frozen-encoder probe vs fine-tuned (test set)

A stronger "base model as-is" baseline than a random head. We load `distilbert-base-uncased` as a **bare encoder** (`AutoModel`) — no classification head, so nothing is randomly initialized and there's no `MISSING` warning. We freeze it, mean-pool its last hidden states into one feature vector per plot, and fit a cheap **logistic-regression head** (one per genre) on the **train** split, then score it on **test**.

This answers a sharper question than "did training do anything": *was full fine-tuning of the encoder worth it over simply probing the frozen pretrained features?* The cell prints a side-by-side table (`base_probe` vs `fine_tuned` with the per-metric `improvement`) and a one-line **verdict** on macro-F1. A small or negative delta would mean the frozen features already carry most of the signal and the extra fine-tuning isn't buying much — a reason to reconsider before the push cell below.

In [ ]:
# "Base model as-is" baseline: a linear probe on the FROZEN pretrained encoder.
# Loading distilbert-base-uncased with AutoModel gives a bare encoder (no
# classification head), so there are no randomly-initialized weights and no
# "MISSING" warning — we use the pretrained features exactly as they come.
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from transformers import AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
collate = DataCollatorWithPadding(tokenizer)


@torch.no_grad()
def encode_features(ds_split):
    """Mean-pool the frozen encoder's last hidden states into one vector per row."""
    loader = DataLoader(
        ds_split.remove_columns("labels"),
        batch_size=BATCH_SIZE * 2,
        collate_fn=collate,
    )
    chunks = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16,
                            enabled=(device.type == "cuda")):
            hidden = encoder(**batch).last_hidden_state          # (B, T, H)
        mask = batch["attention_mask"].unsqueeze(-1).float()      # (B, T, 1)
        pooled = (hidden.float() * mask).sum(1) / mask.sum(1).clamp(min=1)
        chunks.append(pooled.cpu().numpy())
    return np.concatenate(chunks)


X_train = encode_features(train_ds)
X_test = encode_features(test_ds)
y_train = np.array(train_ds["labels"]).astype(int)
y_test = np.array(test_ds["labels"]).astype(int)

# Cheap linear head on the frozen features: one logistic regression per genre.
probe = OneVsRestClassifier(LogisticRegression(max_iter=1000), n_jobs=-1)
probe.fit(X_train, y_train)
probe_preds = probe.predict(X_test).astype(int)  # thresholds each genre at 0.5

probe_overall = {
    "f1_macro": f1_score(y_test, probe_preds, average="macro", zero_division=0),
    "f1_micro": f1_score(y_test, probe_preds, average="micro", zero_division=0),
    "f1_samples": f1_score(y_test, probe_preds, average="samples", zero_division=0),
}

# Compare the frozen-encoder probe against the fine-tuned test numbers (cell above).
comparison = pd.DataFrame(
    {
        "metric": list(test_overall.keys()),
        "base_probe": [probe_overall[k] for k in test_overall],
        "fine_tuned": [test_overall[k] for k in test_overall],
    }
)
comparison["improvement"] = comparison["fine_tuned"] - comparison["base_probe"]

print("=== Frozen-encoder linear probe vs fine-tuned (TEST) ===")
print(comparison.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Verdict on macro-F1: did full fine-tuning beat a cheap probe on frozen features?
delta = test_overall["f1_macro"] - probe_overall["f1_macro"]
improved = delta > 0
print(
    f"\nmacro-F1: {probe_overall['f1_macro']:.4f} (frozen probe) -> "
    f"{test_overall['f1_macro']:.4f} (fine-tuned)  |  delta {delta:+.4f}"
)
print(
    "IMPROVED ✅ — full fine-tuning beat the frozen-encoder probe on the test set."
    if improved
    else "NOT IMPROVED ⚠️ — fine-tuning did not beat a cheap probe on frozen "
    "features; the extra training may not be worth it."
)


## Push to Hub (final step, optional)

Runs only when `PUSH_TO_HUB = True` in the config above. Saves the best checkpoint + tokenizer to `./final`, writes a model card with the test eval numbers and per-genre table, and uploads the folder to `kolodkin/imdb-genre-distilbert`. **This is the only cell that writes to Hugging Face.**


In [ ]:
if not PUSH_TO_HUB:
    print("PUSH_TO_HUB is False — skipping push to Hugging Face.")
else:
    import textwrap
    from huggingface_hub import HfApi

    per_genre_md = test_per_genre.to_markdown(index=False, floatfmt=".4f")
    labels_md = ", ".join(f"`{l}`" for l in LABELS)

    model_card = textwrap.dedent(f"""\
        ---
        license: mit
        library_name: transformers
        base_model: {MODEL_NAME}
        datasets:
        - kolodkin/imdb-wikipedia-enriched
        pipeline_tag: text-classification
        tags:
        - text-classification
        - multi-label
        - genre-classification
        - imdb
        language:
        - en
        ---

        # IMDb Plot → Genre Classifier

        Fine-tuned `{MODEL_NAME}` for multi-label genre classification on movie/show plot summaries from [`kolodkin/imdb-wikipedia-enriched`](https://huggingface.co/datasets/kolodkin/imdb-wikipedia-enriched).

        ## Labels

        {labels_md}

        ## Test results

        | metric | value |
        |---|---|
        | macro-F1 | {test_overall['f1_macro']:.4f} |
        | micro-F1 | {test_overall['f1_micro']:.4f} |
        | samples-F1 | {test_overall['f1_samples']:.4f} |

        95% bootstrap CI on macro-F1: [{lo:.4f}, {hi:.4f}] (1000 resamples).

        ### Per-genre (test)

        """) + per_genre_md + textwrap.dedent(f"""

        ## Usage

        ```python
        from transformers import pipeline

        clf = pipeline("text-classification", model="{HF_REPO}", top_k=None)
        clf("A young wizard discovers a magical school of witchcraft.")
        ```

        ## Training

        - Multi-label stratified 80 / 10 / 10 split (seed {SEED}), deduped on normalized plot hash
        - {NUM_EPOCHS} epochs, lr {LR}, batch size {BATCH_SIZE}, max length {MAX_LENGTH}
        - Best checkpoint selected by val macro-F1
        - Source notebook: https://github.com/kolodkin/samples/blob/main/imdb-genre-distilbert/notebook.ipynb
        """)

    trainer.save_model("./final")
    tokenizer.save_pretrained("./final")
    with open("./final/README.md", "w") as f:
        f.write(model_card)

    api = HfApi()
    api.create_repo(HF_REPO, exist_ok=True, repo_type="model")
    api.upload_folder(
        folder_path="./final",
        repo_id=HF_REPO,
        commit_message=f"Fine-tune {MODEL_NAME} on {DATASET_NAME}",
    )
    print(f"\nPushed to: https://huggingface.co/{HF_REPO}")
